In [0]:
!pip install openpyxl pandas sentence_transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd

In [0]:
df1 = pd.read_excel("/Volumes/hackathon/hack_data/datasets/2021.xlsx").drop(['Round', 'Quota'], axis=1)
df2 = pd.read_excel("/Volumes/hackathon/hack_data/datasets/2022.xlsx").drop(['Round', 'Quota'], axis=1)
df3 = pd.read_excel("/Volumes/hackathon/hack_data/datasets/2023.xlsx").drop(['Round', 'Quota'], axis=1)
df4 = pd.read_excel("/Volumes/hackathon/hack_data/datasets/2024.xlsx").drop(['Round', 'Quota'], axis=1)
df5 = pd.read_excel("/Volumes/hackathon/hack_data/datasets/2025.xlsx").drop(['Quota'], axis=1)

df1.dropna(inplace=True)
df2.dropna(inplace=True)
df3.dropna(inplace=True)
df4.dropna(inplace=True)
df5.dropna(inplace=True)

In [0]:
df1

,Year,Institute,Academic Program Name,Seat Type,Gender,Opening Rank,Closing Rank
0,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Gender-Neutral,8471,12396
1,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Female-only,16998,21029
2,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Gender-Neutral,1603,1760
3,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Female-only,3536,3536
4,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OBC-NCL,Gender-Neutral,3044,4260
...,...,...,...,...,...,...,...
9173,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",OPEN,Gender-Neutral,59431,68295
9174,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",EWS,Gender-Neutral,10427,11089
9175,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",OBC-NCL,Gender-Neutral,21224,21577
9176,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",SC,Gender-Neutral,8907,11617


In [0]:
df5['Year'] = 2025
df5

,Institute,Academic Program Name,Seat Type,Gender,Opening Rank,Closing Rank,Year
0,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Gender-Neutral,10922,16156,2025
1,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Female-only (including Supernumerary),19820,23960,2025
2,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Gender-Neutral,2214,2346,2025
3,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Female-only (including Supernumerary),3688,3698,2025
4,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OBC-NCL,Gender-Neutral,4402,5414,2025
...,...,...,...,...,...,...,...
11939,Shri G. S. Institute of Technology and Science...,Electronics and Instrumentation Engineering (4...,OPEN,Gender-Neutral,48831,51031,2025
11940,Shri G. S. Institute of Technology and Science...,Electronics and Telecommunication Engineering ...,OPEN,Gender-Neutral,35170,41294,2025
11941,Shri G. S. Institute of Technology and Science...,Industrial and Production Engineering (4 Years...,OPEN,Gender-Neutral,67155,71193,2025
11942,Shri G. S. Institute of Technology and Science...,"Information Technology (4 Years, Bachelor of T...",OPEN,Gender-Neutral,28578,32619,2025


In [0]:
def prepare_rag_data(df):
    rules = {
        'Indian Institute of Technology': 'IIT',
        'National Institute of Technology': 'NIT',
        'Indian Institutes of Information Technology': 'IIIT'
    }
    
    for keyword, abbr in rules.items():
        # Mask ensures we don't process rows that already contain the abbreviation (e.g., "IIT")
        mask = df['Institute'].str.contains(keyword, na=False) & ~df['Institute'].str.contains(abbr, na=False)
        
        # Regex explanation:
        # (keyword) captures the base name (Group 1)
        # (.*) captures everything after it, e.g., " Madras" (Group 2)
        # \g<0> represents the entire original match, and we append (abbr + Group 2) to it.
        df.loc[mask, 'Institute'] = df.loc[mask, 'Institute'].str.replace(
            rf'({keyword})(.*)', 
            rf'\g<0> ({abbr}\2)', 
            regex=True
        )
        
    # Vectorized creation of highly contextual statements for RAG
    df['RAG_Context'] = (
        "In " + df['Year'].astype(str) + ", admission to the " + df['Academic Program Name'] + 
        " program at " + df['Institute'] + " under the " + df['Seat Type'] + " category for " + 
        df['Gender'] + " candidates had an opening rank of " + df['Opening Rank'].astype(str) + 
        " and a closing rank of " + df['Closing Rank'].astype(str) + "."
    )
    return df

# Apply the transformation to all 5 dataframes efficiently
dfs = [df1, df2, df3, df4, df5]
for df in dfs:
    prepare_rag_data(df)

In [0]:
df1['Institute'].unique()

array(['Indian Institute of Technology Bhubaneswar (IIT Bhubaneswar)',
       'Indian Institute of Technology Bombay (IIT Bombay)',
       'Indian Institute of Technology Mandi (IIT Mandi)',
       'Indian Institute of Technology Delhi (IIT Delhi)',
       'Indian Institute of Technology Indore (IIT Indore)',
       'Indian Institute of Technology Kharagpur (IIT Kharagpur)',
       'Indian Institute of Technology Hyderabad (IIT Hyderabad)',
       'Indian Institute of Technology Jodhpur (IIT Jodhpur)',
       'Indian Institute of Technology Kanpur (IIT Kanpur)',
       'Indian Institute of Technology Madras (IIT Madras)',
       'Indian Institute of Technology Gandhinagar (IIT Gandhinagar)',
       'Indian Institute of Technology Patna (IIT Patna)',
       'Indian Institute of Technology Roorkee (IIT Roorkee)',
       'Indian Institute of Technology (ISM) Dhanbad (IIT (ISM) Dhanbad)',
       'Indian Institute of Technology Ropar (IIT Ropar)',
       'Indian Institute of Technology (BHU

In [0]:
# List of target institutes as they appear after your previous transformation
target_institutes = [
    'Indian Institute of Technology Madras (IIT Madras)', 
    'Indian Institute of Technology Bombay (IIT Bombay)', 
    'Indian Institute of Technology Delhi (IIT Delhi)', 
    'Indian Institute of Technology Kharagpur (IIT Kharagpur)', 
    'Indian Institute of Technology Kanpur (IIT Kanpur)', 
    'Indian Institute of Technology Roorkee (IIT Roorkee)', 
    'Indian Institute of Technology Guwahati (IIT Guwahati)'
]

# 1. Filter each dataframe and store them in a list
filtered_dfs = [df[df['Institute'].isin(target_institutes)] for df in [df1, df2, df3, df4, df5]]

# 2. Merge them top-to-bottom into a single dataframe
df_merged = pd.concat(filtered_dfs, ignore_index=True)

# Generate counts for each institute in the merged dataframe
institute_counts = df_merged['Institute'].value_counts()

# Display the counts to verify
print("Counts for each Institute in df_merged:")
print(institute_counts)
    
# Optional: verify the result
print(f"Total rows in merged dataframe: {len(df_merged)}")
print(df_merged['Institute'].unique())

# Check for any missing institutes from your target list
missing = set(target_institutes) - set(df_merged['Institute'].unique())
if not missing:
    print("\nVerification Successful: All target institutes are included.")
else:
    print(f"\nWarning: The following institutes are missing from the dataframe: {missing}")

Counts for each Institute in df_merged:
Institute
Indian Institute of Technology Kharagpur (IIT Kharagpur)    1626
Indian Institute of Technology Delhi (IIT Delhi)            1027
Indian Institute of Technology Roorkee (IIT Roorkee)         966
Indian Institute of Technology Bombay (IIT Bombay)           913
Indian Institute of Technology Madras (IIT Madras)           809
Indian Institute of Technology Kanpur (IIT Kanpur)           794
Indian Institute of Technology Guwahati (IIT Guwahati)       647
Name: count, dtype: int64
Total rows in merged dataframe: 6782
['Indian Institute of Technology Bombay (IIT Bombay)'
 'Indian Institute of Technology Delhi (IIT Delhi)'
 'Indian Institute of Technology Kharagpur (IIT Kharagpur)'
 'Indian Institute of Technology Kanpur (IIT Kanpur)'
 'Indian Institute of Technology Madras (IIT Madras)'
 'Indian Institute of Technology Roorkee (IIT Roorkee)'
 'Indian Institute of Technology Guwahati (IIT Guwahati)']

Verification Successful: All target instit

In [0]:
df_merged.to_csv("/Volumes/hackathon/hack_data/datasets/merged_data.csv", index=False)

In [0]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, models
from tqdm import tqdm

# 1. Determine optimal device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Build the custom pipeline
base_model = models.Transformer('sentence-transformers/all-MiniLM-L6-v2')
pooling_layer = models.Pooling(base_model.get_word_embedding_dimension())
dense_layer = models.Dense(in_features=pooling_layer.get_sentence_embedding_dimension(), 
                           out_features=384)

# Passing device here ensures the weights are immediately moved to the target hardware
model = SentenceTransformer(modules=[base_model, pooling_layer, dense_layer], device=device)

# 3. Model Warm-up & Strict Dimension Verification
print("Warming up model and verifying tensors...")
dummy_emb = model.encode(["Initialize computation graph"], convert_to_numpy=True)
verified_dim = dummy_emb.shape[1]

# This hard stop ensures we never process the dataframe if the dimensions are wrong
if verified_dim != 384:
    raise ValueError(f"CRITICAL ERROR: Model output dimension is {verified_dim}, expected 1024.")

print(f"Model fully loaded and initialized on {device.upper()}. Confirmed Output Dimension: {verified_dim}")

# 4. Extract texts
sentences = df_merged['RAG_Context'].tolist()
batch_size = 64
embeddings_list = []

# 5. Generate Embeddings
print(f"Generating embeddings for {len(sentences)} rows...")
for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding"):
    batch = sentences[i : i + batch_size]
    batch_emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    embeddings_list.append(batch_emb)

# 6. Finalize Dataframe
df_merged['Embeddings'] = list(np.vstack(embeddings_list))

# 7. Final Sanity Check
if not df_merged['Embeddings'].isnull().any():
    print("✅ Success! Dataframe completely populated with 384-dim embeddings.")
    df_merged.to_csv("/Volumes/hackathon/hack_data/datasets/data_with_embeddings.csv", index=False)
else:
    print("❌ Verification failed: Null values detected in embeddings column.")

import pandas as pd
import numpy as np
import chromadb
import shutil
import os
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma

# 1. Custom LangChain Wrapper 
class CustomDatabricksEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model
        
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()
        
    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True).tolist()[0]

lc_embedder = CustomDatabricksEmbeddings(model)  # Ensure this model produces 768-dim embeddings

# 2. Custom parser for space-delimited numpy strings
def parse_numpy_string(emb_str):
    clean_str = emb_str.replace('[', '').replace(']', '').strip()
    return [float(x) for x in clean_str.split()]

# 3. Load Data
df = pd.read_csv("/Volumes/hackathon/hack_data/datasets/data_with_embeddings.csv")
if isinstance(df['Embeddings'].iloc[0], str):
    print("Parsing space-delimited embedding strings...")
    df['Embeddings'] = df['Embeddings'].apply(parse_numpy_string)

# 4. Prepare Inputs
documents = df['RAG_Context'].tolist()
embeddings = df['Embeddings'].tolist()
ids = [str(i) for i in range(len(df))]
metadata_cols = ['Year', 'Institute', 'Academic Program Name', 'Seat Type', 'Gender']
metadatas = df[metadata_cols].to_dict('records')

# ==========================================
# NEW: 5. The Databricks Storage Workaround
# ==========================================
local_db_path = "/tmp/chroma_db"
volume_db_path = "/Volumes/hackathon/hack_data/chroma_db"

# If an older database exists on the Volume, bring it to local memory first
if os.path.exists(volume_db_path) and not os.path.exists(local_db_path):
    print("Loading existing ChromaDB from Volume to local compute...")
    shutil.copytree(volume_db_path, local_db_path)

# Initialize ChromaDB on the local node (where SQLite file locking works)
print("Initializing ChromaDB locally...")
client = chromadb.PersistentClient(path=local_db_path)

# NEW: Delete the old collection to clear out the 768-dim requirement
try:
    client.delete_collection(name="iit_admissions")
    print("🗑️ Deleted old 768-dim collection.")
except Exception:
    pass # If it doesn't exist yet, just move on

# Create a completely fresh collection that will lock onto 384 dimensions
collection = client.get_or_create_collection(name="iit_admissions")

# ==========================================
# 6. Push Data in Batches
# ==========================================
print(f"Total vectors to push: {len(documents)}")

# Get the maximum batch size allowed by your specific ChromaDB version
max_batch_size = client.get_max_batch_size()
print(f"ChromaDB max batch size: {max_batch_size}")

# Loop through the data and push it in chunks
for i in range(0, len(ids), max_batch_size):
    # Calculate the end index for the current slice
    end_idx = min(i + max_batch_size, len(ids))
    
    print(f"Pushing batch from index {i} to {end_idx}...")
    
    collection.upsert(
        ids=ids[i:end_idx],
        embeddings=embeddings[i:end_idx],
        documents=documents[i:end_idx],
        metadatas=metadatas[i:end_idx]
    )

print("✅ Vector insertion complete on local node!")

import os

# ==========================================
# 7. Sync the Local Database back to the Permanent Volume
# ==========================================
volume_db_path = "/Volumes/hackathon/hack_data/datasets/chroma_db"
print(f"Syncing Vector DB permanently to {volume_db_path}...")

def safe_copy_to_volume(src_dir, dst_dir):
    """Copies files byte-by-byte to bypass Databricks Volume metadata and security restrictions."""
    os.makedirs(dst_dir, exist_ok=True)
    
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join(dst_dir, item)
        
        if os.path.isdir(s):
            safe_copy_to_volume(s, d)
        else:
            # Pure binary copy: ignores chown/chmod/utime that crash Databricks Volumes
            with open(s, 'rb') as f_src:
                with open(d, 'wb') as f_dst:
                    f_dst.write(f_src.read())

# Execute the safe copy
safe_copy_to_volume(local_db_path, volume_db_path)

print("✅ Database successfully saved to Databricks Volumes!")

# ==========================================
# 8. HOW TO USE WITH LANGCHAIN LATER
# ==========================================
# Note: Since the PersistentClient is currently attached to /tmp/chroma_db, 
# you should point LangChain to the local path while the cluster is running.
vectorstore = Chroma(
    client=client, 
    collection_name="iit_admissions",
    embedding_function=lc_embedder,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/home/spark-9f9862fc-d99f-42c0-86e4-43/.ipykernel/5360/command-7744929568464120-1538846464:12: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_layer = models.Pooling(base_model.get_word_embedding_dimension())
/home/spark-9f9862fc-d99f-42c0-86e4-43/.ipykernel/5360/command-7744929568464120-1538846464:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dense_layer = models.Dense(in_features=pooling_layer.get_sentence_embedding_dimension(),


Warming up model and verifying tensors...
Model fully loaded and initialized on CPU. Confirmed Output Dimension: 384
Generating embeddings for 6782 rows...


Encoding: 100%|██████████| 106/106 [01:33<00:00,  1.13it/s]


✅ Success! Dataframe completely populated with 384-dim embeddings.


In [0]:
%pip install langchain_core chromadb langchain_chroma
%pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import torch
from sentence_transformers import SentenceTransformer, models
import chromadb
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings

# ==========================================
# 1. Sync Database to Local Compute (Run once per cluster start)
# ==========================================
volume_db_path = "/Volumes/hackathon/hack_data/datasets/chroma_db"
local_db_path = "/tmp/chroma_db"

if not os.path.exists(local_db_path):
    print("Pulling vector database from Volume to local memory...")
    # Use Databricks dbutils to safely copy it back to the high-speed local disk
    dbutils.fs.cp(volume_db_path, f"file:{local_db_path}", recurse=True)

# ==========================================
# 2. Rebuild Your Custom Embedding Pipeline
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = models.Transformer('sentence-transformers/all-MiniLM-L6-v2')
pooling_layer = models.Pooling(base_model.get_word_embedding_dimension())
dense_layer = models.Dense(in_features=pooling_layer.get_sentence_embedding_dimension(), out_features=384)
model = SentenceTransformer(modules=[base_model, pooling_layer, dense_layer], device=device)

class CustomDatabricksEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()
    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True).tolist()[0]

lc_embedder = CustomDatabricksEmbeddings(model)

# ==========================================
# 3. Initialize LangChain Vector Store
# ==========================================
client = chromadb.PersistentClient(path=local_db_path)
vectorstore = Chroma(
    client=client,
    collection_name="iit_admissions",
    embedding_function=lc_embedder,
)

# ==========================================
# 4. Configure the Retriever for High Accuracy
# ==========================================
# Using "similarity" search. Setting k=5 retrieves the top 5 most relevant chunks.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# --- Test the Retriever ---
query = "What was the closing rank for Civil Engineering at IIT Madras for Female candidates in 2021?"
print(f"Querying: '{query}'\n")

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Context: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Querying: 'What was the closing rank for Civil Engineering at IIT Madras for Female candidates in 2021?'

--- Result 1 ---
Context: In 2022, admission to the Electrical Engineering with M.Tech. in any of the listed specializations (5 Years, Bachelor and Master of Technology (Dual Degree)) program at Indian Institute of Technology Kharagpur (IIT Kharagpur) under the OPEN category for Female-only candidates had an opening rank of 3970 and a closing rank of 5059.
Metadata: {'Year': 2022, 'Academic Program Name': 'Electrical Engineering with M.Tech. in any of the listed specializations (5 Years, Bachelor and Master of Technology (Dual Degree))', 'Seat Type': 'OPEN', 'Gender': 'Female-only', 'Institute': 'Indian Institute of Technology Kharagpur (IIT Kharagpur)'}

--- Result 2 ---
Context: In 2023, admission to the Electrical Engineering with M.Tech. in any of the listed specializations (5 Years, Bachelor and Master of Technology (Dual Degree)) program at Indian Institute of Technology Khar

/home/spark-9f9862fc-d99f-42c0-86e4-43/.ipykernel/5360/command-7744929568464122-1055722005:24: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_layer = models.Pooling(base_model.get_word_embedding_dimension())
/home/spark-9f9862fc-d99f-42c0-86e4-43/.ipykernel/5360/command-7744929568464122-1055722005:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dense_layer = models.Dense(in_features=pooling_layer.get_sentence_embedding_dimension(), out_features=384)


In [0]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# 1. LOAD MODEL DIRECTLY - No custom pipeline, no random Dense layers!
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

# Verify dimension is natively 384
print(f"Model loaded. Confirmed Output Dimension: {model.get_sentence_embedding_dimension()}")

# 2. Extract texts (Assuming you have df_merged loaded in memory)
sentences = df_merged['RAG_Context'].tolist()
batch_size = 64
embeddings_list = []

# 3. Generate valid, meaningful embeddings
print(f"Generating embeddings for {len(sentences)} rows...")
for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding"):
    batch = sentences[i : i + batch_size]
    batch_emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    embeddings_list.append(batch_emb)

# 4. Finalize Dataframe
df_merged['Embeddings'] = list(np.vstack(embeddings_list))

# 5. Overwrite the corrupted CSV
df_merged.to_csv("/Volumes/hackathon/hack_data/datasets/data_with_embeddings.csv", index=False)
print("✅ Success! Clean 384-dim embeddings saved.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/home/spark-9f9862fc-d99f-42c0-86e4-43/.ipykernel/5360/command-7744929568464129-2548338846:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Confirmed Output Dimension: {model.get_sentence_embedding_dimension()}")


Model loaded. Confirmed Output Dimension: 384
Generating embeddings for 6782 rows...


Encoding: 100%|██████████| 106/106 [01:13<00:00,  1.44it/s]


✅ Success! Clean 384-dim embeddings saved.


In [0]:
import pandas as pd
import chromadb
import os
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma

# 1. Custom LangChain Wrapper (Using the directly loaded model)
class CustomDatabricksEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()
    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True).tolist()[0]

lc_embedder = CustomDatabricksEmbeddings(model)

def parse_numpy_string(emb_str):
    clean_str = emb_str.replace('[', '').replace(']', '').strip()
    return [float(x) for x in clean_str.split()]

# 2. Load the newly generated clean Data
df = pd.read_csv("/Volumes/hackathon/hack_data/datasets/data_with_embeddings.csv")
if isinstance(df['Embeddings'].iloc[0], str):
    df['Embeddings'] = df['Embeddings'].apply(parse_numpy_string)

documents = df['RAG_Context'].tolist()
embeddings = df['Embeddings'].tolist()
ids = [str(i) for i in range(len(df))]
metadata_cols = ['Year', 'Institute', 'Academic Program Name', 'Seat Type', 'Gender']
metadatas = df[metadata_cols].to_dict('records')

# 3. Setup Local ChromaDB
local_db_path = "/tmp/chroma_db"
client = chromadb.PersistentClient(path=local_db_path)

# CRITICAL: Delete the scrambled collection
try:
    client.delete_collection(name="iit_admissions")
    print("🗑️ Deleted old scrambled collection.")
except Exception:
    pass

collection = client.get_or_create_collection(name="iit_admissions")

# 4. Push in Batches
max_batch_size = client.get_max_batch_size()
for i in range(0, len(ids), max_batch_size):
    end_idx = min(i + max_batch_size, len(ids))
    collection.upsert(
        ids=ids[i:end_idx], embeddings=embeddings[i:end_idx],
        documents=documents[i:end_idx], metadatas=metadatas[i:end_idx]
    )

# 5. Safe Copy to Volume
volume_db_path = "/Volumes/hackathon/hack_data/datasets/chroma_db"
def safe_copy_to_volume(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for item in os.listdir(src_dir):
        s, d = os.path.join(src_dir, item), os.path.join(dst_dir, item)
        if os.path.isdir(s): safe_copy_to_volume(s, d)
        else:
            with open(s, 'rb') as f_src, open(d, 'wb') as f_dst:
                f_dst.write(f_src.read())

safe_copy_to_volume(local_db_path, volume_db_path)
print("✅ Clean Database successfully saved to Databricks Volumes!")

🗑️ Deleted old scrambled collection.
✅ Clean Database successfully saved to Databricks Volumes!


In [0]:
# Initialize LangChain Vector Store
vectorstore = Chroma(
    client=client,
    collection_name="iit_admissions",
    embedding_function=lc_embedder,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [0]:
# Test it again!
query = "What was the closing rank for Civil Engineering at IIT Madras in 2022?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Context: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

--- Result 1 ---
Context: In 2021, admission to the Civil Engineering (4 Years, Bachelor of Technology) program at Indian Institute of Technology Madras (IIT Madras) under the OPEN category for Female-only candidates had an opening rank of 8145 and a closing rank of 11569.
Metadata: {'Seat Type': 'OPEN', 'Academic Program Name': 'Civil Engineering (4 Years, Bachelor of Technology)', 'Gender': 'Female-only', 'Institute': 'Indian Institute of Technology Madras (IIT Madras)', 'Year': 2021}

--- Result 2 ---
Context: In 2023, admission to the Civil Engineering (4 Years, Bachelor of Technology) program at Indian Institute of Technology Madras (IIT Madras) under the OPEN category for Female-only candidates had an opening rank of 7746 and a closing rank of 9032.
Metadata: {'Gender': 'Female-only', 'Seat Type': 'OPEN', 'Institute': 'Indian Institute of Technology Madras (IIT Madras)', 'Academic Program Name': 'Civil Engineering (4 Years, Bachelor of Technology)', 'Year': 2023}

--- Result 3 --